# Midterm Project for COMS 4740: Analyzing Representation Bias Under Controlled Class Imbalance (CIFAR-10)

## Overview

This project investigates the impact of class imbalance on the representations learned by a convolutional neural network (CNN) trained on the CIFAR-10 dataset. We will create controlled class imbalance scenarios and analyze how the learned representations differ across classes, particularly focusing on underrepresented classes. The project will involve training a CNN on the imbalanced dataset, extracting features from the penultimate layer, and visualizing these representations using dimensionality reduction techniques like t-SNE or PCA. We will also evaluate the model's performance on the test set to understand how class imbalance affects generalization.

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
import matplotlib.pyplot as plt
import numpy as np
import random
# import copy
# import time

from torch.utils.data import DataLoader, Subset
from torch import optim
from torchvision import transforms, datasets
from scipy.special import softmax
from scipy.optimize import minimize
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split

In [4]:
# Set up GPU

if torch.cuda.is_available():
    #args.device = torch.device("cuda")
    cudnn.benchmark = True
    device = "cuda:0"
else:
    device = "cpu"
print(f'device: {device}')

!nvidia-smi

device: cpu
zsh:1: command not found: nvidia-smi


In [5]:
def set_random_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed )
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.cuda.manual_seed_all(seed)
    random.seed(seed)

set_random_seed()

## Load CIFAR-10 Dataset

In [6]:
# Load CIFAR-10 dataset
data_mean = (0.4914, 0.4822, 0.4465)
data_std = (0.2023, 0.1994, 0.2010)
transform_train = transforms.Compose([
    #transforms.RandomCrop(32, padding=4),
    #transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(data_mean, data_std), ])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(data_mean, data_std), ])

train_data = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
test_set = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

cali_indices, test_indices = train_test_split(
    range(len(test_set)),
    test_size=0.5,
    stratify=test_set.targets, # Stratified sampling
)

cali_data = Subset(test_set, cali_indices)
test_data = Subset(test_set, test_indices)

num_classes = 10
batch_size = 128

# Dataloader
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
cali_loader = DataLoader(cali_data, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)
print(f'Training data length: {len(train_data)}, calibration data length: {len(cali_data)}, test data length: {len(test_data)}')

# Get a single batch of data
train_images, train_labels = next(iter(train_loader))
cali_images, cali_labels = next(iter(cali_loader))
test_images, test_labels = next(iter(test_loader))

print(f'Training batch shape: {train_images.shape}, calibration batch shape: {cali_images.shape}, testing batch shape: {test_images.shape}')

100.0%
/Users/raghavkaashyap/Desktop/Project-COMS4740/.venv/lib/python3.14/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Training data length: 50000, calibration data length: 5000, test data length: 5000
Training batch shape: torch.Size([128, 3, 32, 32]), calibration batch shape: torch.Size([128, 3, 32, 32]), testing batch shape: torch.Size([128, 3, 32, 32])


In [ ]:
# Set up class names for CIFAR-10
class_names = ('plane', 'car', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck')

images_to_plot = {}
for image, label in train_data:
    if label not in images_to_plot:
        images_to_plot[label] = image
    if len(images_to_plot) == 10:
        break

# Create example figures
fig, axes = plt.subplots(2, 5, figsize=(8, 5))
fig.suptitle('Example Images from Each CIFAR-10 Class', fontsize=16)

for i, ax in enumerate(axes.flat):
    if i < len(class_names):
        img_tensor = images_to_plot[i]
        img_unnormalized = img_tensor * torch.tensor(data_std).view(3, 1, 1) + torch.tensor(data_mean).view(3, 1, 1)
        img_unnormalized = img_unnormalized.numpy().transpose((1, 2, 0))
        img_unnormalized = np.clip(img_unnormalized, 0, 1)

        ax.imshow(img_unnormalized)
        ax.set_title(class_names[i])
        ax.axis('off')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

## Controlled Imbalance Setup

Creating balanced/imbalanced training sets and comparing training outcomes.

In [ ]:
from sklearn.metrics import confusion_matrix

def create_imbalance(dataset, class_ratios):
    """Create a subset with per-class retention ratios."""
    targets = np.array(dataset.targets)
    indices = []

    for cls in range(10):
        cls_idx = np.where(targets == cls)[0]
        np.random.shuffle(cls_idx)
        n_samples = int(len(cls_idx) * class_ratios.get(cls, 1.0))
        n_samples = max(1, n_samples)
        indices.extend(cls_idx[:n_samples].tolist())

    return Subset(dataset, indices)

def subset_targets(subset):
    """Return labels for a torch Subset backed by CIFAR-10."""
    base_targets = np.array(subset.dataset.targets)
    return base_targets[np.array(subset.indices)]

def get_class_weights_from_subset(subset, num_classes=10):
    """Inverse-frequency class weights for weighted cross-entropy."""
    targets = subset_targets(subset)
    counts = np.bincount(targets, minlength=num_classes).astype(np.float32)
    counts = np.maximum(counts, 1.0)
    weights = 1.0 / counts
    weights = weights / weights.sum() * num_classes
    return torch.tensor(weights, dtype=torch.float32)

def show_class_distribution(targets, title):
    counts = np.bincount(targets, minlength=10)
    plt.figure(figsize=(7, 3.5))
    plt.bar(np.arange(10), counts)
    plt.xticks(np.arange(10), class_names, rotation=45)
    plt.title(title)
    plt.ylabel("Samples")
    plt.tight_layout()
    plt.show()

balanced_ratios = {i: 1.0 for i in range(10)}
mild_ratios = {0: 1.0, 1: 1.0, 2: 0.5, 3: 0.5, 4: 0.5, 5: 0.5, 6: 1.0, 7: 1.0, 8: 1.0, 9: 1.0}
severe_ratios = {0: 1.0, 1: 1.0, 2: 0.2, 3: 0.2, 4: 0.2, 5: 0.2, 6: 1.0, 7: 1.0, 8: 1.0, 9: 1.0}

imbalanced_train_data = create_imbalance(train_data, severe_ratios)

balanced_targets = np.array(train_data.targets)
imbalanced_targets = subset_targets(imbalanced_train_data)

show_class_distribution(balanced_targets, "Balanced Training Distribution")
show_class_distribution(imbalanced_targets, "Severely Imbalanced Training Distribution")
print(f"Balanced samples: {len(train_data)}")
print(f"Imbalanced samples: {len(imbalanced_train_data)}")

## CNN + Training/Evaluation Helpers

These reusable helpers support all three required experiment settings.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 8 * 8, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

def train_model(model, train_loader, criterion, optimizer, epochs=5):
    model.train()
    for epoch in range(epochs):
        running_loss, total, correct = 0.0, 0, 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            preds = logits.argmax(dim=1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()

        avg_loss = running_loss / max(1, len(train_loader))
        acc = 100.0 * correct / max(1, total)
        print(f"Epoch {epoch + 1}/{epochs} - loss: {avg_loss:.4f}, train acc: {acc:.2f}%")

def evaluate_model(model, data_loader, num_classes=10):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            preds = logits.argmax(dim=1)

            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    cm = confusion_matrix(all_labels, all_preds, labels=np.arange(num_classes))
    overall_acc = (all_preds == all_labels).mean()
    per_class_acc = cm.diagonal() / np.maximum(cm.sum(axis=1), 1)

    return overall_acc, per_class_acc, cm

def plot_confusion_matrix(cm, title):
    plt.figure(figsize=(6, 5))
    plt.imshow(cm, cmap="Blues")
    plt.title(title)
    plt.xlabel("Predicted class")
    plt.ylabel("True class")
    plt.xticks(np.arange(10), class_names, rotation=45)
    plt.yticks(np.arange(10), class_names)
    plt.colorbar()
    plt.tight_layout()
    plt.show()

def run_experiment(name, train_subset, weighted_loss=False, epochs=5, lr=1e-2):
    print(f"\n=== {name} ===")
    train_loader = DataLoader(train_subset, batch_size=128, shuffle=True)
    eval_loader = DataLoader(test_data, batch_size=128, shuffle=False)

    model = SimpleCNN(num_classes=10).to(device)

    if weighted_loss:
        class_weights = get_class_weights_from_subset(train_subset).to(device)
        criterion = nn.CrossEntropyLoss(weight=class_weights)
    else:
        criterion = nn.CrossEntropyLoss()

    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)

    train_model(model, train_loader, criterion, optimizer, epochs=epochs)
    overall_acc, per_class_acc, cm = evaluate_model(model, eval_loader, num_classes=10)

    print(f"Overall accuracy: {overall_acc * 100:.2f}%")
    print("Per-class accuracy:")
    for i, acc in enumerate(per_class_acc):
        print(f"  {class_names[i]}: {acc * 100:.2f}%")

    return {
        "name": name,
        "overall_acc": overall_acc,
        "per_class_acc": per_class_acc,
        "confusion_matrix": cm,
    }

## Experiments and Outputs

Run the three required setups:
1. Balanced + standard loss
2. Imbalanced + standard loss
3. Imbalanced + weighted loss

Compare overall accuracy, per-class accuracy, and confusion matrices.

In [ ]:
balanced_subset = create_imbalance(train_data, balanced_ratios)
imbalanced_subset = create_imbalance(train_data, severe_ratios)

results = []
results.append(run_experiment("Balanced + standard loss", balanced_subset, weighted_loss=False, epochs=5))
results.append(run_experiment("Imbalanced + standard loss", imbalanced_subset, weighted_loss=False, epochs=5))
results.append(run_experiment("Imbalanced + weighted loss", imbalanced_subset, weighted_loss=True, epochs=5))

print("\n=== Summary (Overall Accuracy) ===")
for r in results:
    print(f"{r['name']}: {r['overall_acc'] * 100:.2f}%")

for r in results:
    plot_confusion_matrix(r["confusion_matrix"], f"Confusion Matrix - {r['name']}")

plt.figure(figsize=(10, 4))
x = np.arange(10)
width = 0.25

for i, r in enumerate(results):
    plt.bar(x + i * width, r["per_class_acc"], width=width, label=r["name"])

plt.xticks(x + width, class_names, rotation=45)
plt.ylim(0, 1.0)
plt.ylabel("Per-class accuracy")
plt.title("Per-class Accuracy Across Required Experiments")
plt.legend()
plt.tight_layout()
plt.show()